# HANDS-ON PROJECT: DBSCAN CLUSTERING
## Detecting NYC Airbnb listing hotspots with density-based clustering

**Course:** Data Analysis with Python / Business Analytics  
**Level:** Introductory–Intermediate | **Suggested duration:** 4–6 hours | **Points:** 100  
**Source:** [New York City Airbnb Open Data — Kaggle](https://www.kaggle.com/datasets/dgomonov/new-york-city-airbnb-open-data)  
**File:** `AB_NYC_2019.csv` (2019 snapshot; approximately 48,895 listings, 16 variables; CC0 Public Domain).

### Business scenario
You are a data analyst for a company researching the **geographic concentration of short-term accommodation listings** in New York City. The team wants to find spatially dense listing hotspots, distinguish them from isolated listings, and describe the characteristics of each area. Use **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** to identify hotspots without preselecting the number of clusters.

The geographic density of listings is **not the same as demand, occupancy, profitability, or attractiveness**. The dataset is a historical snapshot, not current market conditions. Business recommendations must be cautious and testable.

## Learning outcomes

By the end of this project, you should be able to:
1. Inspect a real geospatial dataset and identify common quality problems.
2. Explain DBSCAN's `eps`, `min_samples`, **core**, **border**, and **noise** points.
3. Understand why latitude/longitude require an appropriate distance metric (**Haversine distance**).
4. Draw a **k-distance plot** and investigate candidate values of `eps` in kilometres.
5. Compare DBSCAN configurations using cluster count, noise share, and (conditional) silhouette score.
6. Visualize the discovered hotspots and describe them by borough, room type, and nightly price.
7. Evaluate sensitivity to sampling and identify limitations of density-based segmentation.

**Workflow:** Business question → Load CSV → Data audit → EDA → Geographic distances → k-distance diagnostic → DBSCAN → Core/border/noise → Parameter sensitivity → Business profile → Recommendations.

**Grading:** Data quality (10), EDA (10), geographical preparation (15), k-distance (10), DBSCAN visualization and point types (15), tuning/validation (20), profiles (10), robustness/recommendations (10) = **100 points**.

## Part 0 — Environment and original Kaggle data (not graded)

1. Visit the [Kaggle dataset page](https://www.kaggle.com/datasets/dgomonov/new-york-city-airbnb-open-data) and download **`AB_NYC_2019.csv`**.
2. Put the original CSV in the same directory as this notebook, or in a `data/` folder.
3. If the CSV is missing, the loading code *attempts* `kagglehub.dataset_download(...)`. Automatic download needs internet access and may require Kaggle authentication.
4. Install dependencies with `pip install -r requirements.txt` (or install the notebook packages with pip in Google Colab).
5. **Do not use simulated data** for your final project conclusions.

**Computational safeguard:** For accessible student hardware, use a reproducible **8,000-listing sample** for the main models. A sample changes point density, so `eps` and `min_samples` should not be applied to a different sample size without reassessment.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.metrics.pairwise import haversine_distances

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
EARTH_RADIUS_KM = 6371.0088
SAMPLE_N = 8000
OUTPUT = Path('outputs')
OUTPUT.mkdir(exist_ok=True)

In [ ]:
# Load only the original Kaggle CSV; do not generate placeholder observations.
candidates = [Path('AB_NYC_2019.csv'), Path('data/AB_NYC_2019.csv')]
found = next((p for p in candidates if p.is_file()), None)
if found is None:
    try:
        import kagglehub
        kaggle_dir = Path(kagglehub.dataset_download('dgomonov/new-york-city-airbnb-open-data'))
        found = next(iter(kaggle_dir.rglob('AB_NYC_2019.csv')), None)
    except Exception as exc:
        print('Automatic download unavailable:', str(exc)[:180])
if found is None:
    raise FileNotFoundError('Download AB_NYC_2019.csv from Kaggle and put it here or under data/.')
df_raw = pd.read_csv(found)
df_raw.columns = df_raw.columns.str.strip()
required_columns = [
    'id', 'latitude', 'longitude', 'neighbourhood_group', 'neighbourhood',
    'room_type', 'price', 'minimum_nights', 'number_of_reviews', 'availability_365'
]
missing = sorted(set(required_columns) - set(df_raw.columns))
if missing:
    raise ValueError(f'The CSV does not have the expected Kaggle schema: {missing}')
print('Loaded:', found, ' | Shape:', df_raw.shape)
display(df_raw.head(3))

### Key variables and roles

| Column | Meaning | Model role |
|---|---|---|
| `id` | Unique listing identifier | Quality check / export only |
| `latitude`, `longitude` | Listing coordinates in degrees | **Clustering inputs only** |
| `neighbourhood_group` | NYC borough | Interpretation / external description |
| `neighbourhood` | Neighborhood name | Interpretation |
| `room_type` | Entire home/apt, private room, etc. | Interpretation |
| `price` | Nightly advertised price in USD | Interpretation, not distance |
| `minimum_nights` | Minimum stay | Interpretation |
| `number_of_reviews` | Cumulative listing reviews | Interpretation; not direct demand |
| `availability_365` | Available days in a year | Interpretation; not occupancy |

**Design decision:** The main clustering is *spatial* and uses latitude/longitude only. Adding price or review metrics would answer a different question and require a principled mixed-feature distance definition.

## Part 1 — Data audit and cleaning (10 points)

**Task 1.** Explore the dataset. Report its size, data types, missing values, and count of duplicate listing IDs. **Do not confuse repeated latitude/longitude pairs with duplicate listings.**

In [ ]:
# TODO 1: Inspect shape, columns, missing values, and duplicate listing IDs.
# Hints: df_raw.shape, df_raw.dtypes, df_raw.isna().sum(), df_raw['id'].duplicated().sum()

**Task 2.** Create a clean geographic dataset. Remove only records with missing, invalid, or obviously out-of-region coordinates and repeated listing IDs. Count each removal. Explain why missing `reviews_per_month` should **not** exclude a listing from *geographic* clustering.

Use a broad NYC bounding box (latitude `40.45–40.95`, longitude `−74.30–−73.65`) as an auditable geographic check, not as proof that every out-of-range record is incorrect. Report exclusions separately.

In [ ]:
# TODO 2: Clean listing IDs and coordinates. Save a DataFrame called df_clean.
# Hints: drop_duplicates('id'), pd.to_numeric(errors='coerce'), between(...)

**Reflection 1.** What could happen if a listing appears multiple times with slightly different coordinates? Is a distant point necessarily an erroneous listing?

## Part 2 — Exploratory data analysis (10 points)

**Task 3.** Plot the number of listings per borough and the distribution of `room_type`. What does geographic concentration measure, and what *doesn't* it measure?

In [ ]:
# TODO 3: Two count plots (borough and room type).
# Hints: sns.countplot(data=df_clean, x=...), plt.xticks(rotation=...)

**Task 4.** Draw a scatter plot of listing coordinates (`longitude` on the horizontal axis and `latitude` on the vertical axis). Examine the price distribution with a median and a boxplot restricted to the 99th percentile for readability. Do not remove high-price records from geographic clustering merely because of their price.

In [ ]:
# TODO 4: Geographic scatter plot and price summary.
# Hints: plt.scatter(df_clean.longitude, df_clean.latitude), df_clean.price.quantile(.99)

## Part 3 — Geographic distances and reproducible sampling (15 points)

**Task 5.** Select a fixed random sample of at most 8,000 cleaned listings. Verify that the sample includes all five boroughs when they are present in the full dataset. Explain why sampling changes density-based clustering results.

In [ ]:
# TODO 5: Reproducible random sample df_work and print borough proportions.
# Hints: sample(n=min(SAMPLE_N, len(df_clean)), random_state=RANDOM_STATE)

**Task 6.** Convert geographic coordinates from **degrees to radians** and compute the Haversine distance between two sample listings, in kilometres. Briefly explain why `eps=0.2` in the original latitude/longitude coordinates does **not** mean 0.2 km.

Haversine returns **angular distance in radians**; multiply by Earth's approximate mean radius `6371.0088 km` to obtain kilometres. For sklearn's `DBSCAN(metric='haversine')`, pass coordinate values in **radians** and convert `eps_km / EARTH_RADIUS_KM`.

In [ ]:
# TODO 6: Convert geographic coordinates to radians, name the array coords_rad.
# Compute the Haversine separation of the first two sample records in kilometres.
# Hints: np.radians(...), haversine_distances(coords_rad[:1], coords_rad[1:2])

## Part 4 — Understanding density and the k-distance diagnostic (10 points)

**DBSCAN reminder:** A point is **core** when its `eps`-neighborhood contains at least `min_samples` observations, including itself. A **border** point is not core but is assigned to a core-connected cluster. A **noise** point has no such cluster membership. Cluster label `-1` means noise; other cluster numbers have no intrinsic ranking.

**Task 7.** Use `NearestNeighbors(metric='haversine', algorithm='ball_tree')` to compute the distance (in km) to the 12th nearest observation (counting the point itself) for every sampled listing. Sort and plot those distances. A change of slope can suggest a plausible `eps`, but it is **not** an automatic guarantee of an optimal parameter.

In [ ]:
# TODO 7: 12-nearest-neighbor distance plot in kilometres.
# Hints: NearestNeighbors(n_neighbors=12, metric='haversine', algorithm='ball_tree')

**Task 8.** From the k-distance graph, choose a *tentative* `eps` value in kilometres. Explain how increasing `min_samples` or `eps` typically affects core density, number of noise points, and cluster merging. Test a baseline `eps_km=0.20`, `min_samples=12` initially, then revise if needed.

In [ ]:
# TODO 8: Set baseline parameters in physical units (km).
# eps_km = ...
# min_samples = ...

## Part 5 — DBSCAN clustering, hotspots, and point types (15 points)

**Task 9.** Fit `DBSCAN` using the **radian** coordinates and `metric='haversine', algorithm='ball_tree'`. Save labels in `df_work['cluster']`. Report cluster count **excluding** noise and noise percentage.

In [ ]:
# TODO 9: Fit DBSCAN and compute cluster and noise counts.
# Hints: DBSCAN(eps=eps_km/EARTH_RADIUS_KM, min_samples=min_samples, metric='haversine', algorithm='ball_tree')

**Task 10.** Visualize the **eight largest non-noise clusters** with distinct colors; show remaining clusters in gray and noise points in black. Plot in longitude/latitude, but continue using Haversine for distance calculations. Discuss any potentially connected or irregularly shaped clusters.

In [ ]:
# TODO 10: Geographic DBSCAN map with the 8 largest non-noise clusters highlighted.
# Hints: value_counts(), plt.scatter(), label == -1

**Task 11.** Separate **core**, **border**, and **noise** listings using `model.core_sample_indices_` and the output labels. Show counts and shares; explain why border points may be assigned differently in some ambiguous situations.

In [ ]:
# TODO 11: Assign 'core', 'border', and 'noise' point types; summarize frequencies.
# Hints: np.zeros(..., dtype=bool), core_sample_indices_, np.where(...)

## Part 6 — Parameter sensitivity and cautious validation (20 points)

**Task 12.** Experiment with the combinations `eps ∈ {0.12, 0.18, 0.25, 0.35}` km and `min_samples ∈ {8, 12, 20}`. For each fit, record the number of clusters, noise share, core share, largest-cluster share, and (if possible) a **sampled Silhouette Score among non-noise points** using the Haversine metric.

**Important:** A high silhouette score among *only* clustered points can be misleading when the model labels most listings as noise. Report score **alongside** noise percentage and cluster sizes. Some configurations will not have two non-noise clusters; silhouette is then undefined (record `NaN`). To avoid quadratic memory costs, score at most 800 non-noise points using `sample_size`.

In [ ]:
# TODO 12: Grid search over eps_km and min_samples.
# Create a pandas DataFrame named grid_results with columns:
# eps_km, min_samples, n_clusters, noise_share, core_share, largest_cluster_share, silhouette
# Hints: loop, fit_predict(), silhouette_score(..., metric='haversine', sample_size=...)

**Task 13.** Create a figure illustrating the trade-off between noise share and number of clusters for candidate settings; identify at least two configurations worth discussing. Explain why neither the configuration with the highest silhouette score nor the configuration with the fewest noise points must be the best business choice.

In [ ]:
# TODO 13: Plot noise share vs number of clusters; annotate or color by min_samples.
# Hints: sns.scatterplot(data=grid_results, x='noise_share', y='n_clusters', hue='min_samples')

**Task 14.** Select one interpretable configuration based on **cluster structure, noise share, core share, and stability**, rather than silhouette alone. Refit it and create `df_final['cluster']`. Justify your choice in 100–150 words.

The supplied instructor solution uses a transparent **teaching heuristic** (2–25 clusters and at most 65% noise, then higher sampled silhouette; a fallback if no configuration qualifies). Students may choose a different defensible setting.

In [ ]:
# TODO 14: Select best_eps_km and best_min_samples, refit, and create df_final.
# Hints: compare grid_results, DBSCAN(...).fit_predict(coords_rad)

## Part 7 — Spatial hotspot profiles and business interpretation (10 points)

**Task 15.** Describe the largest five non-noise clusters: listing count, borough composition, prevalent room types, median *positive* nightly price, median minimum stay, and median annual availability. Compare with noise listings. Give clusters meaningful **descriptive** names only after examining their profiles.

Do not interpret reviews as verified booking volumes or `availability_365` as realized occupancy. Prices are *advertised*, not necessarily transaction prices.

In [ ]:
# TODO 15: Compute top-5 cluster profiles and a borough cross-tabulation.
# Hints: groupby('cluster').agg(...), pd.crosstab(...), value_counts().idxmax()

## Part 8 — Sensitivity, reporting, and responsible interpretation (10 points)

**Task 16.** Test sample sensitivity. Draw a **second random sample with the same size** from the cleaned dataset and rerun your chosen parameters. Compare cluster count and noise fraction. These samples are *not identical sets of listings*, so do **not** compare their label IDs directly or calculate ARI between unmatched records.

Optional extension: if you want to assess assignment stability for the **same overlapping listing IDs**, align the shared records by `id` before using Adjusted Rand Index (ARI), and discuss noise treatment.

In [ ]:
# TODO 16: Take a different seeded random sample with the same N.
# Rerun the same eps and min_samples, and compare cluster count and noise fraction.
# Hints: df_clean.sample(..., random_state=RANDOM_STATE+1)

### Managerial memo and deliverables

Write a **200–350-word executive memo** addressing:
1. **Where are listing hotspots?** Cite the geographic plot, not an unsupported generalization.
2. Which **2–3 clusters** merit closer market research and why? Use price and room-type profiles only as supporting descriptions.
3. How many listings were marked **noise**, and what operational interpretation *might* this have?
4. What limitations arise from the 2019 snapshot, historical availability, sampled density, fixed `eps`, and the absence of occupancy/profitability data?
5. Propose **two testable follow-up actions**, e.g., investigate local pricing strategies, acquisition opportunities, or host service coverage with *current* ground-truth information.

**Submit:** (a) Completed student `.ipynb`; (b) `outputs/dbscan_nyc_airbnb_clusters.csv`; (c) the executive memo in Markdown/PDF or in the last notebook cell; (d) a brief source citation/link to Kaggle.

**Task to finish:** Export the results (listing `id`, location, cluster) and the parameter comparison table; these exports do not count as a separate numbered task.

In [ ]:
# TODO: Export two CSV files to OUTPUT.
# Hints: df_final[[...]].to_csv(..., index=False), grid_results.to_csv(..., index=False)

## Final reflection questions

1. Why can DBSCAN find irregular shapes whereas K-means favors compact, centroid-defined partitions?
2. In this project, why is `eps` best described in **kilometres**, not in degrees?
3. Why does `-1` mean **unassigned/noise**, not a sixth borough or an inherently low-value listing?
4. Why can a configuration with a high silhouette score still be unhelpful for a market analyst?
5. Why might a single global `eps` be inappropriate in neighborhoods with very different listing densities?
6. How could the conclusions change if you used 2026 data rather than a 2019 snapshot?

**Extension (optional):** Fit DBSCAN separately to each borough and discuss whether borough-specific densities imply different reasonable `eps` values. Do not compare cluster IDs across boroughs as though they had universal meaning.

**Methodology:** scikit-learn [DBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html) and [NearestNeighbors](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html). Dataset: Dgomonov, New York City Airbnb Open Data (2019), Kaggle, CC0.